# Deterministic sampling

Relationship discovery uses row samples and search caps to limit its cost.
This chapter describes how rows are chosen, which passes use each sampler,
and what the samples can miss. Coverage records the sample sizes. Column
cards, keys, duplicates, sortedness, and identity residual checks use full
scans or exact deductions.


## The stratified sampler

`sample_rows(n, cap)` splits the row range into `cap` strata and chooses
one row from each. A fixed arithmetic hash of the stratum index determines
the offset within each stratum. The result is sorted and reproducible for
the same row count and cap.

The first and last strata both contribute rows, so the sample spans the
table. Hashed offsets avoid the fixed spacing of a plain stride, which
can repeatedly select the same part of periodic data, such as one weekday
in a weekly export. They do not guarantee that every periodic pattern is
represented. When the table fits under the cap, every row is used.

The example has 300,000 rows: an `id`, ten random columns, a target, and a
planted lookup pair (`region` → `region_name`). Its coverage block shows
the row counts used by the different passes.


In [ ]:
import numpy as np
import polars as pl

import onceover

rng = np.random.default_rng(3)
n = 300_000
region = rng.integers(0, 5, n)
big = pl.DataFrame(
    {"id": np.arange(n, dtype=np.int64)}
    | {"region": region}
    | {"region_name": [f"r{r}" for r in region]}
    | {f"x{i:02}": rng.random(n) for i in range(10)}
    | {"y": rng.random(n)}
)

report = onceover.profile(big)
for name in ["encoding", "predictability_screen", "functional_dependencies", "dependence_screen"]:
    print(f"{name:24s}", report.coverage[name]["detail"])
    print()

The coverage detail pairs every sample with its population — "20000 rows
(20000 of 300000)" — so a reader always knows which claims were measured
where. On this frame the encoded matrix and the dependence statistics run
over 20,000 stratified rows; the FD screen looks at 26 directed pairs
over 2,048 rows, then verifies its two discovered candidates over a
200,000-row sample; the column cards and keys never sample at all.

## Which pass uses which rows


In [ ]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Pass': ['Column cards, duplicates, single keys, sortedness, dates', 'FD broad screen', 'FD verification', 'Composite-key screen / verification', 'Encoded matrix', 'Predictability screen', 'Decision tables', 'PRISM rules', 'Rule evaluation', 'Association mining', 'Dependence statistics', 'Identity fitting', 'Identity residual check', 'Regimes', 'Duplicate encodings', 'Target shapes', 'Surprises and sentinels', 'Temporal breaks', 'Discontinuities', 'Interactions', 'Hotspots', 'Views', 'Declared expectations'], 'Rows used (`standard` / `deep`)': ['all rows', '2,048', '≤ 200,000 / 500,000', '2,048 screen; ≤ 16 / 24 verified pairs, full data', '≤ 20,000 / 50,000', '≤ 5,000 / 10,000 fitting rows', 'encoded rows', 'encoded rows', '1/5 of sampled rows', '4/5 train, 1/5 holdout', '≤ 20,000 / 50,000', '≤ 20,000', 'all pair-complete rows', '≤ 5,000 / 10,000', '≤ 100,000 / 250,000', 'all predictor–target complete rows', 'all eligible rows within each processed scope', '≤ 10,000 / 50,000 time-ordered rows', '≤ 20,000 / 50,000 complete rows per unit', '128 / 192 partial-dependence points', 'encoded rows', '≤ 20,000 / 50,000 fitting rows; ≤ 2,000 search rows', 'all eligible input rows'], 'Mechanism': ['exact', 'stratified sample of hashed values', 'stratified sample, canonical value bytes', 'hash screen, exact count', 'stratified', 'seeded shuffle, 75/25 holdout per target', '5-fold cross-validation', 'bitset masks over the encoded sample', 'row-id hash partition', 'row-id hash partition', 'stratified, pair-complete', 'stratified', 'exact', '1/5 row-id holdout', 'exact canonical verification on a sampled population', 'same-data shape comparisons', 'full-value scans, bounded candidate/context output', 'training-selected splits and diagnostic holdout', 'deterministic fit/evaluation halves', 'fitted model and empirical background', 'complete rows per combination', 'opt-in greedy partition search', 'bounded reference-calendar enumeration']})
(GT(_doc_table)
 .fmt_markdown(columns=['Pass', 'Rows used (`standard` / `deep`)', 'Mechanism'])
 .tab_header(title='Which pass uses which rows')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


The predictability screen shuffles positions with a seeded generator and
holds out 25% per target. Rule evaluation, nominal association mining, and
regime analysis use fixed hashes of original row indices, with method-specific
partition definitions. Their evaluated sets can overlap and should not be
treated as independent replications.

A model's holdout may participate in early stopping or method selection.
The common rule evaluation can also use rows already seen during
discovery. Evidence records that use through `used_for_selection`; these
scores are diagnostic rather than final independent confirmation.


## What samples can miss

A 20,000-row sample may miss a pattern present in 0.01% of rows, or include
too few examples to detect it. Per-feature shortlisting can drop signals
that appear only through interactions. The FD search stops at its
per-target determinant shortlist. Coverage records sample sizes, caps,
and shortlists. Overrides such as `sample_rows`, `max_predictors`,
`max_pairs`, and budget `deep` increase the corresponding search budgets.

A finding from a sample describes the evaluated rows. If a finding is
missing, increase the budget or test the hypothesis directly. A bounded
search cannot establish that two columns are independent.
